# 歌词采集-QQ音乐

In [1]:
import requests
import re
import json
import os
import time
import pandas as pd
import html
from datetime import datetime
from collections import defaultdict


from collections import Counter

In [2]:
import sys
sys.path.append('..')

# 通用方法

## 时间戳格式化

In [3]:
def format_timestamp(ts, date_format='%Y-%m-%d'):
    """
    自动识别秒或毫秒，并转换为指定格式的字符串
    :param ts: 时间戳 (int 或 float)
    """
    if not ts or ts <= 0:
        return "Unknown"
    
    # 核心逻辑：判断时间戳位数
    # 秒级时间戳目前在 10^9 数量级（10位）
    # 毫秒级时间戳在 10^12 数量级（13位）
    # 我们以 10^11 (11位) 为界限进行区分
    if ts > 100000000000: 
        ts = ts / 1000  # 是毫秒，转换为秒
    
    try:
        dt = datetime.fromtimestamp(ts)
        return dt.strftime(date_format)
    except Exception:
        return "Invalid Date"

## 增量保存到json文件

In [4]:
def save_to_json_list(file_path, song_data):
    """以列表形式保存所有歌曲，避免字典 key 覆盖的问题"""
    data_list = []
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data_list = json.load(f)
                if not isinstance(data_list, list): data_list = []
            except:
                data_list = []

    data_list.append(song_data)

    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data_list, f, ensure_ascii=False, indent=4)

# 按歌手采集曲目

In [5]:
def search_song(keyword, page=0):
    """搜索歌曲并返回歌曲ID"""
    url = "https://c.y.qq.com/soso/fcgi-bin/client_search_cp"
    params = {
        "w": keyword,
        "format": "json",
        "n": 50,
        "p": page,
    }
    headers = {
        "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url, params=params, headers=headers)
    if response.status_code == 200:
        data = json.loads(response.text)
        songs = data["data"]["song"]["list"]
        res = []
        for song in songs:
            res_d = {
                "song_id": song["songid"],
                "song_mid": song["songmid"],
                "song_name": song["songname"],
                "song_subname": song["lyric"],
                "artist_name": ",".join([singer["name"] for singer in song["singer"]]),
                "artist_id": ",".join([str(singer["id"]) for singer in song["singer"]]),
                "artist_mid": ",".join([singer["mid"] for singer in song["singer"]]),
                "album_name": song['albumname'],
                "album_id": song['albumid'],
                "album_mid": song['albummid'],
                "duration": song['interval'],
                "publish_time": song["pubtime"],
            }
            res.append(res_d)
        return res
    return []

In [6]:
def get_songs_data_raw(singger, max_page=1):
    """
    获取歌手歌曲列表
    singer_name: 歌手名称
    max_page: 最大页数, 默认每页50条数据，max_page=曲目总数/50
    """
    qq_songs_list = []
    for page in range(0, max_page):
        print(f'正在获取第{page+1}页数据...')
        res = search_song(singger, page)
        time.sleep(2)
        qq_songs_list.extend(res)
    return qq_songs_list

# 曲目过滤

## OST曲目筛选

In [7]:
# 数据筛选
# 1. artist_name中包含singger
# 2. song_subname中包含书名号，将书名号中的内容保存为新字段: tv_name
def filter_ost_songs(singger, song_list):
    """
    筛选符合条件的歌曲：
    1. 歌手包含 singger
    2. 子标题包含书名号，并提取书名号内容为 tv_name
    """
    filtered_list = []
    # 预编译正则，匹配《 和 》之间最少的内容
    tv_pattern = re.compile(r'《(.*?)》')

    for song in song_list:
        # 条件 1: 校验 artist_name (确保该字段已在之前的解析中生成)
        if singger not in song.get("artist_name", ""):
            continue

        # 条件 2: 校验 song_name，如果”《“在 song_name 中，则跳过
        if "《" in song.get("song_name", ""):
            continue

        # 条件 2.5: 如果 song_subname 中包含 试听，伴奏，则跳过
        if any(keyword in song.get("song_name", "") for keyword in ["试听", "伴奏"]):
            continue
            
        # 条件 3: 校验 song_subname 并在满足时提取 tv_name
        subname = song.get("song_subname", "")
        match = tv_pattern.search(str(subname))
        
        if match:
            # 满足条件，创建新字段并保存
            song["tv_name"] = match.group(1)
            filtered_list.append(song)
            
    return filtered_list

## 按专辑列表

In [8]:
def filter_album_songs(singger, song_list, album_list):
    """
    筛选符合条件的歌曲：
    1. 歌手包含 singger
    2. 专辑包含在 album_list 中
    """
    filtered_list = []

    for song in song_list:
        singers = song.get("artist_name", "")
        album_name = song.get("album_name", "")
        if singger in singers and album_name in album_list:
            filtered_list.append(song)
            
    return filtered_list

## 曲目数据清洗

In [9]:
def clear_song_data(song_data,
                    is_filter_ost=False,
                    is_use_raw_song_name=False):
    """
    清洗歌曲数据
    is_filter_ost: 是否过滤掉OST歌曲
    is_use_raw_song_name: 二次清洗时，是否使用原始歌曲名，例如五月天这种，原始歌曲就有多个版本的，需要在歌名中保留括号内容
    """
    songs_df = pd.DataFrame(song_data)
    if is_use_raw_song_name:
        songs_df['song_name_unique'] = songs_df['song_name']
    else:
        # 分割song_name中的括号
        songs_df['song_name_unique'] = songs_df['song_name'].apply(
            lambda x: x.split('(')[0])
        songs_df['song_name_unique'] = songs_df['song_name_unique'].apply(
            lambda x: x.split('（')[0])
        # 删除前后空格
        songs_df['song_name_unique'] = songs_df['song_name_unique'].apply(
            lambda x: x.strip())
    # 按song_name_unique进行去重
    songs_df = songs_df.drop_duplicates(subset=['song_name_unique'],
                                        keep='first')
    # 发行时间格式化
    songs_df['publish_date'] = songs_df['publish_time'].apply(
        lambda x: format_timestamp(x))
    if is_filter_ost:
        # 二次筛选ost， 新建列 is_ost, 如果song_subname中含 影，剧，曲任意一个字，则is_ost为1，否则为0
        # songs_df['is_ost'] = songs_df['song_subname'].apply(
        #     lambda x: 1 if '影' in str(x) or '剧' in str(x) or '曲' in str(
        #         x) or '片' in str(x) else 0)
        songs_df['is_ost'] = songs_df['song_subname'].apply(
            lambda x: 1 if '影' in str(x) or '剧' in str(x) or '片' in str(x) else 0)
        songs_df = songs_df[songs_df['is_ost'] == 1]
        # songs_df = songs_df
    return songs_df

## 专辑信息清洗

In [10]:
# 专辑数据清洗
def clear_album_data(songs_data):
    album_count = songs_data.groupby(
        ['album_name',
         'album_id'])['album_id'].count().reset_index(name='count')
    album_count = album_count.sort_values(by=['count'], ascending=False)
    # 取count最大值所在行的数据为album_id
    album_id = album_count.drop_duplicates(subset=['album_name'],
                                           keep='first').reset_index(drop=True)
    # 匹配专辑发行日期
    album_date = songs_data[['album_id', 'publish_date']].drop_duplicates(
        subset=['album_id'], keep='first').reset_index(drop=True)
    album_df = album_id[['album_name', 'album_id']].merge(album_date,
                                                          on='album_id',
                                                          how='left')
    songs_data_cleared = songs_data.drop(['album_id', 'publish_date'],
                                         axis=1).copy()
    songs_data_cleared = songs_data_cleared.merge(album_df,
                                                  on='album_name',
                                                  how='left')
    return songs_data_cleared

# 歌词采集

In [11]:
def get_qq_lyric(song_id):
    """根据歌曲ID获取歌词"""
    url = "https://c.y.qq.com/lyric/fcgi-bin/fcg_query_lyric_yqq.fcg"
    params = {
        "nobase64": 1,
        "musicid": song_id,
        "format": "json"
    }
    headers = {
        "Referer": "https://y.qq.com/",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url, params=params, headers=headers)
    if response.status_code == 200:
        lyric_data = response.json()
        return lyric_data.get("lyric", "")
    return "歌词获取失败"

In [12]:
def get_all_songs_lyric(file_path, songs_df):
    for i, row in songs_df.iterrows():
        song_id = row['song_id']
        song_name = row['song_name']
        # 判断是否已采集
        if os.path.exists(file_path):
            df = pd.read_json(file_path)
            songs_had = df['song_id'].tolist()
            if song_id not in songs_had:
                print(song_name)
                lyric_raw = get_qq_lyric(song_id)
                single_res = {
                    'song_id': song_id,
                    'song_name': song_name,
                    'lyric_raw': lyric_raw
                }
                save_to_json_list(file_path, single_res)
                time.sleep(2)
        else:
            print(song_name)
            lyric_raw = get_qq_lyric(song_id)
            single_res = {
                    'song_id': str(song_id),
                    'song_name': song_name,
                    'lyric_raw': lyric_raw
                }
            save_to_json_list(file_path, single_res)
            time.sleep(2)

## 歌词清洗

In [13]:
class QQLyricCleanerV16:
    def __init__(self):
        self.time_tag_pattern = re.compile(r'\[(\d{2,}:?\d{2,}\.\d{2,})\]\s*(.*)')

    def _generate_singer_blacklist(self, singers):
        if not singers: return set()
        # 兼容中英文逗号、斜杠、空格拆分
        names = re.split(r'[,，/ ]', singers)
        blacklist = {'男', '女', '合', '人', '们'}
        for name in names:
            name = name.strip()
            if not name: continue
            blacklist.add(name.upper()) 
            if len(name) > 1:
                blacklist.add(name[0].upper()) # 姓
        return blacklist

    def _preprocess(self, text):
        if not text: return ""
        text = html.unescape(text)
        return text.replace('\r', '').replace('\\n', '\n')

    def get_credits(self, raw_text):
        text = self._preprocess(raw_text)
        res = {"lyricist": "", "composer": "", "arranger": ""}
        mapping = {
            "lyricist": r"(?:词|作词)\s*[:：]\s*([^\n\r\]]+)",
            "composer": r"(?:曲|作曲)\s*[:：]\s*([^\n\r\]]+)",
            "arranger": r"(?:编曲|制作人|Arranger|Program)\s*[:：]\s*([^\n\r\]]+)"
        }
        for key, pat in mapping.items():
            match = re.search(pat, text, re.IGNORECASE)
            if match:
                res[key] = re.split(r'[\[\]]', match.group(1).strip())[0].strip()
        return res

    def process_data(self, raw_lyric, song_id=None, song_name="", singers=""):
        singer_bits = self._generate_singer_blacklist(singers)
        credits = self.get_credits(raw_lyric)
        text = self._preprocess(raw_lyric)
        
        valid_lines = []
        lines = text.split('\n')
        content_line_idx = 0
        
        for line in lines:
            match = self.time_tag_pattern.search(line)
            if not match: continue
            
            ts, content = match.groups()
            content = content.strip()
            if not content: continue

            # --- 核心改进：标题/标题行判定 ---
            clean_content = content.replace(" ", "").upper()
            
            # 识别标题的特征：包含歌曲名、包含歌手名、或者包含大量分隔标点
            contains_singer = any(s in clean_content for s in singer_bits if len(s) > 1)
            is_metadata_structure = any(symbol in content for symbol in ['-', '《', '，', '(', '（'])
            
            # 判定：如果是前几行，且长得像标题或元数据，直接过滤
            if content_line_idx < 3:
                if (song_name and song_name.upper() in clean_content) or contains_singer or is_metadata_structure:
                    content_line_idx += 1
                    continue

            # 权利声明过滤
            if any(w in clean_content for w in ['权利保留', '未经许可', '著作权', '版权', '声明', '提供']):
                content_line_idx += 1
                continue

            # --- 冒号处理逻辑 ---
            if ':' in content or '：' in content:
                sep = ':' if ':' in content else '：'
                prefix, *suffix = content.split(sep, 1)
                prefix_clean = prefix.strip().upper()
                suffix_content = "".join(suffix).strip()

                if prefix_clean in singer_bits:
                    if suffix_content:
                        content = suffix_content
                    else:
                        content_line_idx += 1
                        continue
                else:
                    # 只要不在演唱者白名单里的冒号行，全部删除
                    content_line_idx += 1
                    continue

            # 最终清洗
            c_final = re.sub(r'\s+', '，', content).strip('，')
            if c_final:
                valid_lines.append((ts, c_final))
                content_line_idx += 1

        # 数据组装
        start_time = valid_lines[0][0] if valid_lines else ""
        lyrics_text = "。".join([x[1] for x in valid_lines])
        if lyrics_text: lyrics_text += "。"

        return {
            "song_id": song_id,
            "song_name": song_name,
            "start_time": start_time,
            "has_lyric": 1 if lyrics_text else 0,
            "lyricist": credits["lyricist"],
            "composer": credits["composer"],
            "arranger": credits["arranger"],
            "lyrics_text": lyrics_text
        }

## 清洗main

In [14]:
def clear_and_save_lyric(path_prefix, songs_df):
    lyric_raw = pd.read_json(path_prefix+'raw_lyric_data.json')

    cleaner = QQLyricCleanerV16()
    res_list = []
    for i, row in songs_df.iterrows():
        song_id = row['song_id']
        song_name = row['song_name']
        singers = row.get('artist_name', "")
        lyric_raw_single = lyric_raw[lyric_raw['song_id'] == song_id]['lyric_raw'].values[0]
        single_res = cleaner.process_data(lyric_raw_single, song_id, song_name, singers)
        res_list.append(single_res)
    with open(path_prefix+'cleared_lyric_data.json', 'w', encoding='utf-8') as f:
        json.dump(res_list, f, ensure_ascii=False, indent=4)

# main

## 歌手-按专辑筛选数据

In [214]:
# file_path_prefix = "data/jaychou/"
# singger = "周杰伦"
# max_page = 20

file_path_prefix = "data/mayday/"
singger = "五月天"
max_page = 20

### 曲目采集

In [215]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singger, max_page=max_page)

正在获取第1页数据...
正在获取第2页数据...
正在获取第3页数据...
正在获取第4页数据...
正在获取第5页数据...
正在获取第6页数据...
正在获取第7页数据...
正在获取第8页数据...
正在获取第9页数据...
正在获取第10页数据...
正在获取第11页数据...
正在获取第12页数据...
正在获取第13页数据...
正在获取第14页数据...
正在获取第15页数据...
正在获取第16页数据...
正在获取第17页数据...
正在获取第18页数据...
正在获取第19页数据...
正在获取第20页数据...


In [216]:
df_song_data_raw = pd.DataFrame(song_data_raw)
# 原始曲目保存
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [193]:
# 重新读取数据
df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')

In [194]:
# 曲目筛选，按专辑
# 周杰伦
album_list = [
    "Jay", "范特西", "八度空间", "叶惠美", "七里香", "十一月的萧邦", "依然范特西", "我很忙", "魔杰座", "跨时代", "惊叹号", "十二新作", "周杰伦的床边故事", "最伟大的作品", "哎呦，不错哦"
]
# 五月天
# album_list = [
#     '第一张创作专辑', '爱情万岁', '人生海海', '时光机', '神的孩子都在跳舞', '为爱而生', '后青春期的诗',
#     '第二人生（明日版）', '第二人生（末日版）', '自传', '知足 最真杰作选', '步步 自选作品辑 the Best of 1999-2013'
# ]
song_data_filted = filter_album_songs(singger, song_data_raw_read, album_list)

In [195]:
len(song_data_filted)

199

### 数据验证
检查专辑中歌曲是否存在缺失，将缺失歌曲的数据手工添加到song_data_filted中

In [196]:
song_data_cleared_1 = clear_song_data(song_data_filted, is_use_raw_song_name=False)
song_data_cleared_1

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
0,97773,0039MnYb0qxYhV,晴天,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,叶惠美,8220,000MkMni19ClKG,269,1059580800,晴天,2003-07-31
1,102065756,004Z8Ihr0JIu5s,七里香,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,七里香,20612,003DFRzD192KKD,299,1091462400,七里香,2004-08-03
2,449205,003aAYrm3GE0Ac,稻香,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,魔杰座,36062,002Neh8l0uciQZ,223,1224000000,稻香,2008-10-15
3,410316,002qU5aY3Qu24y,青花瓷,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,我很忙,33021,002eFUFm2XYZ7z,239,1193932800,青花瓷,2007-11-02
4,449198,003cI52o4daJJL,花海,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,魔杰座,36062,002Neh8l0uciQZ,264,1224000000,花海,2008-10-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194,3586267,0010jyte3izshw,四季列车,NaN,周杰伦,4558,0025NhlN2yWrP4,十二新作,194021,003Ow85E3pnoqi,159,1356624000,四季列车,2012-12-28
195,107192077,003uYI3j4EDf5q,土耳其冰淇淋,NaN,周杰伦,4558,0025NhlN2yWrP4,周杰伦的床边故事,1458791,003RMaRI1iFoYd,195,1466697600,土耳其冰淇淋,2016-06-24
196,449202,003suDvd4KLUmF,流浪诗人,NaN,周杰伦,4558,0025NhlN2yWrP4,魔杰座,36062,002Neh8l0uciQZ,169,1224000000,流浪诗人,2008-10-15
197,680283,000ES19W2iTudx,嘻哈空姐,NaN,周杰伦,4558,0025NhlN2yWrP4,跨时代,56705,000bviBl4FjTpO,168,1274112000,嘻哈空姐,2010-05-18


In [197]:
song_data_cleared_1.groupby('album_name')['song_name'].count()

album_name
Jay         10
七里香         10
依然范特西       10
八度空间        10
十一月的萧邦      11
十二新作        12
叶惠美         11
周杰伦的床边故事     9
哎呦，不错哦      12
惊叹号         11
我很忙         10
最伟大的作品       7
范特西         10
跨时代         11
魔杰座         11
Name: song_name, dtype: int64

In [186]:
# 问题专辑
# 周杰伦
album_list_to_fix = ['哎呦，不错哦']
# 五月天
# album_list_to_fix = ['第二人生（明日版）']
song_data_cleared_1[song_data_cleared_1['album_name'] == album_list_to_fix[0]]

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
18,101787870,002u8ZOM4C7QF4,手写的从前,NaN,周杰伦,4558,0025NhlN2yWrP4,哎呦，不错哦,852856,001uqejs3d6EID,297,1419523200,手写的从前,2014-12-26
90,101369814,001Js78a40BZU6,算什么男人,NaN,周杰伦,4558,0025NhlN2yWrP4,哎呦，不错哦,852856,001uqejs3d6EID,288,1419523200,算什么男人,2014-12-26
123,101787873,0026N9sT4aBGcV,听见下雨的声音,NaN,周杰伦,4558,0025NhlN2yWrP4,哎呦，不错哦,852856,001uqejs3d6EID,279,1419523200,听见下雨的声音,2014-12-26
127,101787872,001XPgTm3faHRb,美人鱼,NaN,周杰伦,4558,0025NhlN2yWrP4,哎呦，不错哦,852856,001uqejs3d6EID,219,1419523200,美人鱼,2014-12-26
138,101787866,00162TQ12285h4,天涯过客,NaN,周杰伦,4558,0025NhlN2yWrP4,哎呦，不错哦,852856,001uqejs3d6EID,253,1419523200,天涯过客,2014-12-26
149,101133228,004KpNVr0EdY0v,鞋子特大号,NaN,周杰伦,4558,0025NhlN2yWrP4,哎呦，不错哦,852856,001uqejs3d6EID,221,1419523200,鞋子特大号,2014-12-26
153,101787867,001zqHER0WFQvO,怎么了,NaN,周杰伦,4558,0025NhlN2yWrP4,哎呦，不错哦,852856,001uqejs3d6EID,232,1419523200,怎么了,2014-12-26
154,101787871,003BiKB44LknC0,听爸爸的话,NaN,周杰伦,4558,0025NhlN2yWrP4,哎呦，不错哦,852856,001uqejs3d6EID,263,1419523200,听爸爸的话,2014-12-26
173,101787869,0007w9Eb0lilhZ,我要夏天,NaN,周杰伦,4558,0025NhlN2yWrP4,哎呦，不错哦,852856,001uqejs3d6EID,219,1419523200,我要夏天,2014-12-26
186,101787868,003md41m1MBhMc,一口气全念对,NaN,周杰伦,4558,0025NhlN2yWrP4,哎呦，不错哦,852856,001uqejs3d6EID,158,1419523200,一口气全念对,2014-12-26


In [198]:
# 补充歌曲，并修改专辑信息
# 周杰伦
songs_dict_to_add = {'说好不哭 (with 五月天阿信)' : '最伟大的作品', '不爱我就拉倒': '最伟大的作品', 'Mojito': '最伟大的作品', '等你下课 (with 杨瑞代)': '最伟大的作品', '我是如此相信': '最伟大的作品', '英雄': '周杰伦的床边故事', '一路向北': '十一月的萧邦'}
# 五月天
# songs_dict_to_add = {'生命有一种绝对': '时光机', 'Enrich Your Life': '神的孩子都在跳舞', '垃圾车 (朋友版)': '神的孩子都在跳舞', '温柔 (还你自由版)': '知足 最真杰作选', '入阵曲': '步步 自选作品辑 the Best of 1999-2013', '离开地球表面': '步步 自选作品辑 the Best of 1999-2013', 'OAOA (丢掉名字性别)': '第二人生（末日版）', '我不愿让你一个人': '第二人生（末日版）'}
for i in song_data_raw_read:
    if i['song_name'] in songs_dict_to_add.keys():
        print(i)

{'song_id': 5105986, 'song_mid': '001xd0HI0X9GNq', 'song_name': '一路向北', 'song_subname': '《头文字D》电影插曲', 'artist_name': '周杰伦', 'artist_id': 4558, 'artist_mid': '0025NhlN2yWrP4', 'album_name': 'J III MP3 Player', 'album_id': 14311, 'album_mid': '002MAeob3zLXwZ', 'duration': 294, 'publish_time': 1119542400}
{'song_id': 212877900, 'song_mid': '001J5QJL1pRQYB', 'song_name': '等你下课 (with 杨瑞代)', 'song_subname': nan, 'artist_name': '周杰伦', 'artist_id': 4558, 'artist_mid': '0025NhlN2yWrP4', 'album_name': '等你下课', 'album_id': 3883404, 'album_mid': '003bSL0v4bpKAx', 'duration': 270, 'publish_time': 1516204800}
{'song_id': 237773700, 'song_mid': '001qvvgF38HVc4', 'song_name': '说好不哭 (with 五月天阿信)', 'song_subname': nan, 'artist_name': '周杰伦', 'artist_id': 4558, 'artist_mid': '0025NhlN2yWrP4', 'album_name': '说好不哭（with 五月天阿信）', 'album_id': 7876962, 'album_mid': '002gBTVk4JEE2T', 'duration': 222, 'publish_time': 1568646000}
{'song_id': 247261229, 'song_mid': '001PLl3C4gPSCI', 'song_name': '我是如此相信', 'song_subn

In [199]:
# 手工添加需要补全的歌曲
# 周杰伦
songs_to_add = [{
    'song_id': 105755384,
    'song_mid': '004Qscj80GYhGR',
    'song_name': '英雄',
    'song_subname': '《英雄联盟》中国品牌主题曲',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '周杰伦的床边故事',
    'album_id': 1306793,
    'album_mid': '001uJFiE0tbGGa',
    'duration': 200,
    'publish_time': 1458748800
}, {
    'song_id': 247261229,
    'song_mid': '001PLl3C4gPSCI',
    'song_name': '我是如此相信',
    'song_subname': '《天火》电影主题曲',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 9612009,
    'album_mid': '001hGx1Z0so1YX',
    'duration': 266,
    'publish_time': 1576339200
}, {
    'song_id': 268352018,
    'song_mid': '001glaI72k8BQX',
    'song_name': 'Mojito',
    'song_subname': '',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 12924001,
    'album_mid': '0009C3rp3Kfwg0',
    'duration': 185,
    'publish_time': 1591891200
}, {
    'song_id': 213922043,
    'song_mid': '0031TAKo0095np',
    'song_name': '不爱我就拉倒',
    'song_subname': '',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 4044657,
    'album_mid': '001CnPE31iJ899',
    'duration': 245,
    'publish_time': 1526313600
}, {
    'song_id': 212877900,
    'song_mid': '001J5QJL1pRQYB',
    'song_name': '等你下课 (with 杨瑞代)',
    'song_subname': '',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 3883404,
    'album_mid': '003bSL0v4bpKAx',
    'duration': 270,
    'publish_time': 1516204800
}, {
    'song_id': 237773700,
    'song_mid': '001qvvgF38HVc4',
    'song_name': '说好不哭 (with 五月天阿信)',
    'song_subname': '',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 7876962,
    'album_mid': '002gBTVk4JEE2T',
    'duration': 222,
    'publish_time': 1568646000
}, {
    'song_id': 5105986,
    'song_mid': '001xd0HI0X9GNq',
    'song_name': '一路向北',
    'song_subname': '《头文字D》电影插曲',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': 'J III MP3 Player',
    'album_id': 14311,
    'album_mid': '002MAeob3zLXwZ',
    'duration': 294,
    'publish_time': 1119542400
}]
# 修改专辑名称
songs_to_add_fixed = []
for song in songs_to_add:
    song['album_name'] = songs_dict_to_add[song['song_name']]
    songs_to_add_fixed.append(song)

In [ ]:
# 五月天
songs_to_add = [
    {
        'song_id': 405385,
        'song_mid': '001BAFqt1Ay4Vf',
        'song_name': '离开地球表面',
        'song_subname': '《开心超人》动画电影片尾曲',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '离开地球表面 Jump!',
        'album_id': 32775,
        'album_mid': '002PYDbl3I5L2k',
        'duration': 275,
        'publish_time': 1184860800
    },
    {
        'song_id': 4996096,
        'song_mid': '003Xy9E32vvMLe',
        'song_name': '入阵曲',
        'song_subname': '《兰陵王》电视剧主题曲',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '兰陵王 电视剧原声带',
        'album_id': 431765,
        'album_mid': '002adz882rV5uh',
        'duration': 209,
        'publish_time': 1377792000
    },
    {
        'song_id': 4932058,
        'song_mid': '003PaRAX3j5wJk',
        'song_name': '生命有一种绝对',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '摇滚本事 电影音乐原声带',
        'album_id': 96335,
        'album_mid': '0015r2I31enfaR',
        'duration': 239,
        'publish_time': 1038672000
    },
    {
        'song_id': 4830242,
        'song_mid': '000PoJAV4NPMzW',
        'song_name': '温柔 (还你自由版)',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '音乐电影-五月之恋',
        'album_id': 96361,
        'album_mid': '001ntd0y01uQ4g',
        'duration': 426,
        'publish_time': 1088611200
    },
    {
        'song_id': 1056504,
        'song_mid': '002Sv0dp3T3p5U',
        'song_name': 'OAOA (丢掉名字性别)',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '第二人生（末日版）',
        'album_id': 90142,
        'album_mid': '000IPRft1LSCqL',
        'duration': 282,
        'publish_time': 1323964800
    },
    {
        'song_id': 4834459,
        'song_mid': '002nqyCb1bUnk6',
        'song_name': 'Enrich Your Life',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': 'Enrich Your Life',
        'album_id': 62890,
        'album_mid': '0006oAnx03zXUC',
        'duration': 166,
        'publish_time': 1096560000
    },
    {
        'song_id': 4932456,
        'song_mid': '002qi7L00Cpb73',
        'song_name': '垃圾车 (朋友版)',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '神的孩子都在跳舞',
        'album_id': 96368,
        'album_mid': '002plCgA0zOyYF',
        'duration': 243,
        'publish_time': 1099238400
    },
    {'song_id': 519403016, 'song_mid': '000B69Qg0S8WUF', 'song_name': '我不愿让你一个人', 'song_subname': '《今夜一起为爱鼓掌》电视剧插曲', 'artist_name': '五月天', 'artist_id': 74, 'artist_mid': '000Sp0Bz4JXH0o', 'album_name': '今夜一起为爱鼓掌 电视剧原声带', 'album_id': 56247634, 'album_mid': '001Jhk1t0SC1FZ', 'duration': 265, 'publish_time': 1727625600}

]
# 修改专辑名称
songs_to_add_fixed = []
for song in songs_to_add:
    song['album_name'] = songs_dict_to_add[song['song_name']]
    songs_to_add_fixed.append(song)

### 专辑名称手工修改

In [ ]:
# 五月天
album_to_fix_dict = {
    '第二人生（末日版）': '第二人生',
    '第二人生（明日版）': '第二人生',
    '步步 自选作品辑 the Best of 1999-2013': '步步 自选作品辑',
}

In [200]:
song_data_filted.extend(songs_to_add_fixed)


In [ ]:
# 修正专辑名称，需要时运行
for i in song_data_filted:
    if i['album_name'] in album_to_fix_dict:
        i['album_name'] = album_to_fix_dict[i['album_name']]

In [201]:
len(song_data_filted)

206

### 二次清洗

In [202]:
# song_data_filted.extend(songs_to_add)
# 五月天歌曲名使用原始歌曲名，其他则为is_use_raw_song_name=False
song_data_cleared_2 = clear_song_data(song_data_filted, is_use_raw_song_name=False)
song_data_cleared_2

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
0,97773,0039MnYb0qxYhV,晴天,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,叶惠美,8220,000MkMni19ClKG,269,1059580800,晴天,2003-07-31
1,102065756,004Z8Ihr0JIu5s,七里香,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,七里香,20612,003DFRzD192KKD,299,1091462400,七里香,2004-08-03
2,449205,003aAYrm3GE0Ac,稻香,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,魔杰座,36062,002Neh8l0uciQZ,223,1224000000,稻香,2008-10-15
3,410316,002qU5aY3Qu24y,青花瓷,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,我很忙,33021,002eFUFm2XYZ7z,239,1193932800,青花瓷,2007-11-02
4,449198,003cI52o4daJJL,花海,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,魔杰座,36062,002Neh8l0uciQZ,264,1224000000,花海,2008-10-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,268352018,001glaI72k8BQX,Mojito,,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,12924001,0009C3rp3Kfwg0,185,1591891200,Mojito,2020-06-12
202,213922043,0031TAKo0095np,不爱我就拉倒,,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,4044657,001CnPE31iJ899,245,1526313600,不爱我就拉倒,2018-05-15
203,212877900,001J5QJL1pRQYB,等你下课 (with 杨瑞代),,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,3883404,003bSL0v4bpKAx,270,1516204800,等你下课,2018-01-18
204,237773700,001qvvgF38HVc4,说好不哭 (with 五月天阿信),,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,7876962,002gBTVk4JEE2T,222,1568646000,说好不哭,2019-09-16


In [203]:
song_data_cleared_2.groupby('album_name')['song_name'].count()

album_name
Jay         10
七里香         10
依然范特西       10
八度空间        10
十一月的萧邦      12
十二新作        12
叶惠美         11
周杰伦的床边故事    10
哎呦，不错哦      12
惊叹号         11
我很忙         10
最伟大的作品      12
范特西         10
跨时代         11
魔杰座         11
Name: song_name, dtype: int64

In [ ]:
song_data_cleared_2[song_data_cleared_2['album_name'] == '第二人生']

In [ ]:
# 需要手工删除的歌
songs_to_drop = [
    'T1 21 31 21 (Bonus Track)', '知足 (乐团版)', '拥抱 (2013新录制作品)',
    '温柔 (2013Remix版)', '憨人 (Live)'
]
# 删除song_data_cleared中song_name在songs_to_drop的行
song_data_cleared = song_data_cleared_2[~song_data_cleared_2['song_name'].
                                         isin(songs_to_drop)]
song_data_cleared

In [205]:
song_data_cleared_final = clear_album_data(song_data_cleared_2)
song_data_cleared_final

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,publish_time,song_name_unique,album_id,publish_date
0,97773,0039MnYb0qxYhV,晴天,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,叶惠美,000MkMni19ClKG,269,1059580800,晴天,8220,2003-07-31
1,102065756,004Z8Ihr0JIu5s,七里香,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,七里香,003DFRzD192KKD,299,1091462400,七里香,20612,2004-08-03
2,449205,003aAYrm3GE0Ac,稻香,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,魔杰座,002Neh8l0uciQZ,223,1224000000,稻香,36062,2008-10-15
3,410316,002qU5aY3Qu24y,青花瓷,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,我很忙,002eFUFm2XYZ7z,239,1193932800,青花瓷,33021,2007-11-02
4,449198,003cI52o4daJJL,花海,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,魔杰座,002Neh8l0uciQZ,264,1224000000,花海,36062,2008-10-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,268352018,001glaI72k8BQX,Mojito,,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,0009C3rp3Kfwg0,185,1591891200,Mojito,28791467,2022-07-14
158,213922043,0031TAKo0095np,不爱我就拉倒,,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,001CnPE31iJ899,245,1526313600,不爱我就拉倒,28791467,2022-07-14
159,212877900,001J5QJL1pRQYB,等你下课 (with 杨瑞代),,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,003bSL0v4bpKAx,270,1516204800,等你下课,28791467,2022-07-14
160,237773700,001qvvgF38HVc4,说好不哭 (with 五月天阿信),,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,002gBTVk4JEE2T,222,1568646000,说好不哭,28791467,2022-07-14


In [206]:
song_data_cleared_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

### 歌词采集

In [207]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', song_data_cleared_final)

夜曲
枫
发如雪
手写的从前
珊瑚海
黑色毛衣
算什么男人
浪漫手机
听见下雨的声音
飘移
美人鱼
天涯过客
麦芽糖
鞋子特大号
四面楚歌
怎么了
听爸爸的话
逆鳞
我要夏天
蓝色风暴
一口气全念对
窃爱
阳明山
一路向北


### 歌词清洗

In [208]:
clear_and_save_lyric(file_path_prefix, song_data_cleared_final)

     song_id song_name                                          lyric_raw
0      97773        晴天  [ti&#58;晴天]&#10;[ar&#58;周杰伦]&#10;[al&#58;叶惠美]&...
1  102065756       七里香  [ti&#58;七里香]&#10;[ar&#58;周杰伦]&#10;[al&#58;七里香]...
2     449205        稻香  [ti&#58;稻香]&#10;[ar&#58;周杰伦]&#10;[al&#58;魔杰座]&...
3     410316       青花瓷  [ti&#58;青花瓷]&#10;[ar&#58;周杰伦]&#10;[al&#58;我很忙]...
4     449198        花海  [ti&#58;花海]&#10;[ar&#58;周杰伦]&#10;[al&#58;魔杰座]&...


## 歌手-不需要按专辑筛选数据

In [15]:
# file_path_prefix = "data/liuyuning/"
# singger = "刘宇宁"
# max_page = 14

file_path_prefix = "data/newyear/"
singger = "过年"
max_page = 3

### 曲目采集

In [16]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singger, max_page=max_page)

正在获取第1页数据...
正在获取第2页数据...
正在获取第3页数据...


In [17]:
df_song_data_raw = pd.DataFrame(song_data_raw)
# 原始曲目保存
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [18]:
# 重新读取数据
df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')

In [19]:
song_data_raw_read

[{'song_id': 213054628,
  'song_mid': '000MDaNK0krdFb',
  'song_name': '好运来',
  'song_subname': nan,
  'artist_name': '祖海',
  'artist_id': '7352',
  'artist_mid': '002m2h2n4Veqcj',
  'album_name': '好运来',
  'album_id': 3908735,
  'album_mid': '001KHxdj0upU5l',
  'duration': 213,
  'publish_time': 1074268800},
 {'song_id': 106686506,
  'song_mid': '000I3Ih84QiS9s',
  'song_name': '恭喜发财',
  'song_subname': nan,
  'artist_name': '刘德华',
  'artist_id': '163',
  'artist_mid': '003aQYLo2x8izP',
  'album_name': '春节音乐',
  'album_id': 199630,
  'album_mid': '003dNHXg186ZPk',
  'duration': 202,
  'publish_time': 1165939200},
 {'song_id': 4826060,
  'song_mid': '002NCGTk1dhWpG',
  'song_name': '过年好',
  'song_subname': nan,
  'artist_name': '天孪兄弟',
  'artist_id': '15319',
  'artist_mid': '000absdi49XwcT',
  'album_name': '天生一对',
  'album_id': 429916,
  'album_mid': '000MjDNI1CW5iU',
  'duration': 186,
  'publish_time': 1369670400},
 {'song_id': 228213715,
  'song_mid': '000kdH8V2ieyeS',
  'song_name

### 清洗

In [145]:
song_data_filted = filter_ost_songs(singger, song_data_raw_read)

In [146]:
song_data_cleared = clear_song_data(song_data_filted,
                                    is_filter_ost=True,
                                    is_use_raw_song_name=True)
song_data_cleared

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,tv_name,song_name_unique,publish_date,is_ost
1,629395245,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,风过留痕 影视原声带,81722850,003Dq7Cs1NdHv8,265,1770084000,风过留痕,荣光,2026-02-03,1
2,578702351,001QwWqP38bXCs,烽月,电视剧《折腰》情感主题曲/片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,折腰 影视原声专辑,67523527,001NFTqM3zNUv2,265,1747411200,折腰,烽月,2025-05-17,1
3,370388317,001l38Gx3sl3kP,寻一个你,《苍兰诀》电视剧温情主题曲,刘宇宁,2241311,001Iu4Dv1NzRCD,苍兰诀 东方幻想影视原声带,33455648,001c5tj84NH6dH,267,1660010400,苍兰诀,寻一个你,2022-08-09,1
4,298883373,002kOvBZ3eQChP,天问,《山河令》网剧主题曲,刘宇宁,2241311,001Iu4Dv1NzRCD,山河令 网剧音乐原声大碟,18540389,003HQCiV10f8Bw,221,1614009600,山河令,天问,2021-02-23,1
5,285903302,004a6xSN489klN,当遇见你,《冰糖炖雪梨》电视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,当遇见你,15738588,0027ZeN63fdE93,191,1583769600,冰糖炖雪梨,当遇见你,2020-03-10,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,401100858,0009Ba6M3Rd0kp,望道,《望道》电影同名主题曲,"刘宇宁,8点组乐团","2241311,1162539","001Iu4Dv1NzRCD,004OkF1U0FkWIb",望道,36285879,001SGs7l1BP05L,196,1678982400,望道,望道,2023-03-17,1
174,335548330,0032iLRJ3YH8fN,风起时再见,《迷雾追踪》影视剧主题曲,刘宇宁,2241311,001Iu4Dv1NzRCD,迷雾追踪 影视原声带,33427051,003V3TEN24Hj29,260,1607616000,迷雾追踪,风起时再见,2020-12-11,1
176,264776262,002rs2Tk3fYVGh,要替我幸福,《暖暖，请多指教》电视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,暖暖，请多指教 影视原声带,12373795,003eJOh54YBBK1,242,1589472000,暖暖，请多指教,要替我幸福,2020-05-15,1
177,297559879,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,斗罗大陆 史兰客七怪音乐专辑,17500078,000uo5Ig1624LG,272,1613354400,斗罗大陆,苍穹之下,2021-02-15,1


In [147]:
song_data_cleared.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

### 歌词采集

In [58]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', song_data_cleared)

荣光
烽月
寻一个你
天问
当遇见你
奉上
让酒
长风谣
失去后一切如常
纸片人
独爱
万两
毒酒
那朵花
我爱你，中国
消失的记忆
我只愿
万剑不改
就在江湖之上
琉璃
热辣滚烫
无心生大梦
孤弈长安
不沐春风不遇你
有多少爱可以重来
你说爱情啊
惟愿
无双
一爱如故
缘圈
昨日少年
直到时间尽头
不忘
梦华
世世
二中中二日记
如约
烟火星辰
凌云寂
我爱的这个世界
踏苍穹
泼墨
别梦寒
一番星
千里江山
以你之名
逍遥仙
长安
年长
云字诀
风吹过
共度
一往无畏
寻常
宁愿
朝暮
不敢逢春
梦魇之后
问千年
感应
心动
你是我所有
清晖
如果爱记得
可追
这一路
挺好个人呐
眷恋
踏夜升明
风衣
向死而生
引力
隐侠
装模作样
爱是不放手
狂风袭来
匆匆
寻岸
意气趁年少
莫问前程
No Matter (唯一的回答)
余生只想握紧你的手
果如
灼心
长相诺
You are the one
我只愿朝着光
本可以
天光
浩渺
如梦亦如你
无华
反客为主
莫悲歌
同生
阳光、海浪、我和你
爱了很久
断缘诀
仰望晴空
同袍 (国语版)
选择去爱你
孤舟
心悠悠
愿重逢
没那么难
世界只有我们
初升
生命之书
唯一的光
穿越回忆奔向你
浪
你是我的星空
一生有多远
余生
我曾经穿过黑夜
呢喃
无尽海
虎啸春来
追光
少有人走的路
浮生
念
拂晓
遥远的相似
去温暖的地方
最浪漫的忘记
作者
就让过去都过去
友情岁月 (兄弟版)
天行健
老朋友
微世界
我愿意
专属蓝天
好运歌
爱情之所以
如果爱回应
望道
风起时再见
要替我幸福
苍穹之下
年长 (15秒试听版片段)
青年有为
不敢逢春 (试听版)
烽月 (伴奏)


### 歌词清洗

In [153]:
song_data_cleared.info()

<class 'pandas.core.frame.DataFrame'>
Index: 142 entries, 1 to 179
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   song_id           142 non-null    int64 
 1   song_mid          142 non-null    object
 2   song_name         142 non-null    object
 3   song_subname      142 non-null    object
 4   artist_name       142 non-null    object
 5   artist_id         142 non-null    object
 6   artist_mid        142 non-null    object
 7   album_name        142 non-null    object
 8   album_id          142 non-null    int64 
 9   album_mid         142 non-null    object
 10  duration          142 non-null    int64 
 11  publish_time      142 non-null    int64 
 12  tv_name           142 non-null    object
 13  song_name_unique  142 non-null    object
 14  publish_date      142 non-null    object
 15  is_ost            142 non-null    int64 
dtypes: int64(5), object(11)
memory usage: 18.9+ KB


In [164]:
clear_and_save_lyric(file_path_prefix, song_data_cleared)

# 关键词-采集

In [20]:
# file_path_prefix = "data/liuyuning/"
# singger = "刘宇宁"
# max_page = 14

file_path_prefix = "data/newyear/"
singger = "过年"
max_page = 3

### 曲目采集

In [21]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singger, max_page=max_page)

正在获取第1页数据...
正在获取第2页数据...
正在获取第3页数据...


In [22]:
df_song_data_raw = pd.DataFrame(song_data_raw)
# 原始曲目保存
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [23]:
# 重新读取数据
df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')

In [24]:
song_data_raw_read

[{'song_id': 213054628,
  'song_mid': '000MDaNK0krdFb',
  'song_name': '好运来',
  'song_subname': nan,
  'artist_name': '祖海',
  'artist_id': '7352',
  'artist_mid': '002m2h2n4Veqcj',
  'album_name': '好运来',
  'album_id': 3908735,
  'album_mid': '001KHxdj0upU5l',
  'duration': 213,
  'publish_time': 1074268800},
 {'song_id': 106686506,
  'song_mid': '000I3Ih84QiS9s',
  'song_name': '恭喜发财',
  'song_subname': nan,
  'artist_name': '刘德华',
  'artist_id': '163',
  'artist_mid': '003aQYLo2x8izP',
  'album_name': '春节音乐',
  'album_id': 199630,
  'album_mid': '003dNHXg186ZPk',
  'duration': 202,
  'publish_time': 1165939200},
 {'song_id': 4826060,
  'song_mid': '002NCGTk1dhWpG',
  'song_name': '过年好',
  'song_subname': nan,
  'artist_name': '天孪兄弟',
  'artist_id': '15319',
  'artist_mid': '000absdi49XwcT',
  'album_name': '天生一对',
  'album_id': 429916,
  'album_mid': '000MjDNI1CW5iU',
  'duration': 186,
  'publish_time': 1369670400},
 {'song_id': 228213715,
  'song_mid': '000kdH8V2ieyeS',
  'song_name

In [27]:
song_data_cleared = clear_song_data(song_data_raw_read,
                                    is_filter_ost=False,
                                    is_use_raw_song_name=False)
song_data_cleared

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
0,213054628,000MDaNK0krdFb,好运来,NaN,祖海,7352,002m2h2n4Veqcj,好运来,3908735,001KHxdj0upU5l,213,1074268800,好运来,2004-01-17
1,106686506,000I3Ih84QiS9s,恭喜发财,NaN,刘德华,163,003aQYLo2x8izP,春节音乐,199630,003dNHXg186ZPk,202,1165939200,恭喜发财,2006-12-13
2,4826060,002NCGTk1dhWpG,过年好,NaN,天孪兄弟,15319,000absdi49XwcT,天生一对,429916,000MjDNI1CW5iU,186,1369670400,过年好,2013-05-28
3,228213715,000kdH8V2ieyeS,张灯结彩,NaN,"王二妮,阿宝","41449,5001","003lTJ1u3lA6hv,003oUwJ54CMqTT",猪福-孔雀群星贺新年,6094085,0045t5Q32NT70Q,244,1548950400,张灯结彩,2019-02-01
4,121641128,002NmMFi0YBMuS,欢乐中国年,NaN,孙悦,4448,004Tu0h03OnCTF,NaN,0,NaN,260,0,欢乐中国年,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,228067708,000RxHNy1WqRH1,银河系DISCO,《疯狂的外星人》电影宣传主题曲,火箭少女101,2169599,000Rh4DK4MmxIU,银河系Disco,6048031,001bO2VY23yfoo,226,1548604800,银河系DISCO,2019-01-28
144,105461811,001RQ9vB3RxxFW,喜庆临门,NaN,孙家山,166790,002sKdn31xAF4E,喜庆临门,1267543,000x9lRc2dUvXK,183,1451404800,喜庆临门,2015-12-30
145,200551951,0005TCNp4EOcEW,红红火火又一年,NaN,望海高歌,67405,001r9gSh4HW230,红红火火又一年,1829157,001nqm7L1QgOQh,223,1485014400,红红火火又一年,2017-01-22
147,456693581,004SZkCy1bU208,一年又一年,NaN,土豆王国小乐队,1398384,000vFOyZ2pFpVl,一年又一年,44723902,003FOIB523CM1Y,208,1703001600,一年又一年,2023-12-20


In [28]:
# 删除song_name_unique中有标点符号，有数字,的歌
def filter_song_name(song_name):
    # 如果歌名中有标点符号或数字，返回False
    if re.search(r'[^\w\s]', song_name) or re.search(r'\d', song_name):
        return False
    return True

song_data_cleared = song_data_cleared[song_data_cleared['song_name_unique'].apply(filter_song_name)]
# 按song_name_unique去重
song_data_cleared = song_data_cleared.drop_duplicates(subset=['song_name_unique'], keep='first')
song_data_cleared


,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
0,213054628,000MDaNK0krdFb,好运来,NaN,祖海,7352,002m2h2n4Veqcj,好运来,3908735,001KHxdj0upU5l,213,1074268800,好运来,2004-01-17
1,106686506,000I3Ih84QiS9s,恭喜发财,NaN,刘德华,163,003aQYLo2x8izP,春节音乐,199630,003dNHXg186ZPk,202,1165939200,恭喜发财,2006-12-13
2,4826060,002NCGTk1dhWpG,过年好,NaN,天孪兄弟,15319,000absdi49XwcT,天生一对,429916,000MjDNI1CW5iU,186,1369670400,过年好,2013-05-28
3,228213715,000kdH8V2ieyeS,张灯结彩,NaN,"王二妮,阿宝","41449,5001","003lTJ1u3lA6hv,003oUwJ54CMqTT",猪福-孔雀群星贺新年,6094085,0045t5Q32NT70Q,244,1548950400,张灯结彩,2019-02-01
4,121641128,002NmMFi0YBMuS,欢乐中国年,NaN,孙悦,4448,004Tu0h03OnCTF,NaN,0,NaN,260,0,欢乐中国年,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,228067708,000RxHNy1WqRH1,银河系DISCO,《疯狂的外星人》电影宣传主题曲,火箭少女101,2169599,000Rh4DK4MmxIU,银河系Disco,6048031,001bO2VY23yfoo,226,1548604800,银河系DISCO,2019-01-28
144,105461811,001RQ9vB3RxxFW,喜庆临门,NaN,孙家山,166790,002sKdn31xAF4E,喜庆临门,1267543,000x9lRc2dUvXK,183,1451404800,喜庆临门,2015-12-30
145,200551951,0005TCNp4EOcEW,红红火火又一年,NaN,望海高歌,67405,001r9gSh4HW230,红红火火又一年,1829157,001nqm7L1QgOQh,223,1485014400,红红火火又一年,2017-01-22
147,456693581,004SZkCy1bU208,一年又一年,NaN,土豆王国小乐队,1398384,000vFOyZ2pFpVl,一年又一年,44723902,003FOIB523CM1Y,208,1703001600,一年又一年,2023-12-20


In [29]:
# 手动删除 东北民谣，银河系DISCO
song_data_cleared = song_data_cleared[~song_data_cleared['song_name_unique'].isin(['东北民谣', '银河系DISCO'])]
song_data_cleared

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
0,213054628,000MDaNK0krdFb,好运来,NaN,祖海,7352,002m2h2n4Veqcj,好运来,3908735,001KHxdj0upU5l,213,1074268800,好运来,2004-01-17
1,106686506,000I3Ih84QiS9s,恭喜发财,NaN,刘德华,163,003aQYLo2x8izP,春节音乐,199630,003dNHXg186ZPk,202,1165939200,恭喜发财,2006-12-13
2,4826060,002NCGTk1dhWpG,过年好,NaN,天孪兄弟,15319,000absdi49XwcT,天生一对,429916,000MjDNI1CW5iU,186,1369670400,过年好,2013-05-28
3,228213715,000kdH8V2ieyeS,张灯结彩,NaN,"王二妮,阿宝","41449,5001","003lTJ1u3lA6hv,003oUwJ54CMqTT",猪福-孔雀群星贺新年,6094085,0045t5Q32NT70Q,244,1548950400,张灯结彩,2019-02-01
4,121641128,002NmMFi0YBMuS,欢乐中国年,NaN,孙悦,4448,004Tu0h03OnCTF,NaN,0,NaN,260,0,欢乐中国年,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,108706119,004esi930dmRrc,好年头好兆头,NaN,卓依婷,6351,002UILPL4dGoEz,祝福1,881291,0000DmIY3nG4Qn,362,1069603200,好年头好兆头,2003-11-24
144,105461811,001RQ9vB3RxxFW,喜庆临门,NaN,孙家山,166790,002sKdn31xAF4E,喜庆临门,1267543,000x9lRc2dUvXK,183,1451404800,喜庆临门,2015-12-30
145,200551951,0005TCNp4EOcEW,红红火火又一年,NaN,望海高歌,67405,001r9gSh4HW230,红红火火又一年,1829157,001nqm7L1QgOQh,223,1485014400,红红火火又一年,2017-01-22
147,456693581,004SZkCy1bU208,一年又一年,NaN,土豆王国小乐队,1398384,000vFOyZ2pFpVl,一年又一年,44723902,003FOIB523CM1Y,208,1703001600,一年又一年,2023-12-20


In [30]:
song_data_cleared.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

In [34]:
aaa = """过年啦 过年啦

国泰民安中国年

我用成语拜新年


祝大家



一帆风顺二龙腾飞




三羊开泰四季平安




五福临门六六大顺




七星高照八方来财




吉星高照张灯结彩

辞旧迎新阖家欢乐

出口成章恭贺新禧

年年有余岁岁平安




一祝爷爷奶奶福如东海



二祝姥姥姥爷寿比南山



三祝爸爸妈妈万事如意



四祝叔叔阿姨好运连连



五祝老师同学意气风发



六祝自我超越勇往直前




啊 万象更新又一年

千言万语说不完

满怀憧憬共祝愿

祝愿大家团团圆圆

啊 万象更新又一年

千言万语说不完

满怀憧憬共祝愿

祝愿大家团团圆圆




吉星高照张灯结彩

辞旧迎新阖家欢乐

出口成章恭贺新禧

年年有余岁岁平安




一帆风顺二龙腾飞



三羊开泰四季平安



五福临门六六大顺



七星高照八方来财



九九同心十全十美



百尺竿头千里之志




万事如意 年复一年

万事如意年复一年

啊 万象更新又一年

千言万语说不完

满怀憧憬共祝愿

祝愿大家团团圆圆

啊 万象更新又一年

千言万语说不完

满怀憧憬共祝愿

祝愿大家团团圆圆"""

In [39]:
# 删除aaa中的空格换为句号，文字间的首个换行符替换为句号，其他换行符删除
bbb = re.sub(r'\n+', '。', aaa.replace(' ', ''))

bbb

'过年啦过年啦。国泰民安中国年。我用成语拜新年。祝大家。一帆风顺二龙腾飞。三羊开泰四季平安。五福临门六六大顺。七星高照八方来财。吉星高照张灯结彩。辞旧迎新阖家欢乐。出口成章恭贺新禧。年年有余岁岁平安。一祝爷爷奶奶福如东海。二祝姥姥姥爷寿比南山。三祝爸爸妈妈万事如意。四祝叔叔阿姨好运连连。五祝老师同学意气风发。六祝自我超越勇往直前。啊万象更新又一年。千言万语说不完。满怀憧憬共祝愿。祝愿大家团团圆圆。啊万象更新又一年。千言万语说不完。满怀憧憬共祝愿。祝愿大家团团圆圆。吉星高照张灯结彩。辞旧迎新阖家欢乐。出口成章恭贺新禧。年年有余岁岁平安。一帆风顺二龙腾飞。三羊开泰四季平安。五福临门六六大顺。七星高照八方来财。九九同心十全十美。百尺竿头千里之志。万事如意年复一年。万事如意年复一年。啊万象更新又一年。千言万语说不完。满怀憧憬共祝愿。祝愿大家团团圆圆。啊万象更新又一年。千言万语说不完。满怀憧憬共祝愿。祝愿大家团团圆圆'

### 歌词采集

In [31]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', song_data_cleared)

好运来
恭喜发财
过年好
张灯结彩
欢乐中国年
中国喜事
恭喜呀恭喜
春风十里报新年
发财发福中国年
团团圆圆
福气要来到
拜新年
成语拜新年
大貔貅
大吉大利中国年
新年快乐
春意红包
恭喜发财红包拿来
祝新岁
今年胜旧年
财源滚滚来
过年的歌
红包摇
吉祥中国年
新年大吉
红红火火中国年
祝福你
恭喜恭喜
吉祥年
过年啦 (童声版)
八星报喜贺贺喜
样样红
恭喜呀 恭喜呀 新年好
好运全都来
四喜临门喜迎春
一个好年
财神驾到
八仙齐拜年
一年更比一年好
接财神
生意兴隆
春到了
喜乐年华
正月初一过新年
新年好
新年这一刻
天气预爆
皆大欢喜
欢喜过新年
拜年歌
红包
万事如意
新年的钟声
有钱没钱回家过年
大家恭喜
万事胜胜意
恭祝大家新年好
喜事齐来
新的一年
五福金鸡喜满堂
开局迎好运
好年头好兆头
喜庆临门
红红火火又一年
一年又一年
中国吉祥


### 歌词清洗

In [32]:
song_data_cleared.info()

<class 'pandas.core.frame.DataFrame'>
Index: 66 entries, 0 to 148
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   song_id           66 non-null     int64 
 1   song_mid          66 non-null     object
 2   song_name         66 non-null     object
 3   song_subname      7 non-null      object
 4   artist_name       66 non-null     object
 5   artist_id         66 non-null     object
 6   artist_mid        66 non-null     object
 7   album_name        65 non-null     object
 8   album_id          66 non-null     int64 
 9   album_mid         65 non-null     object
 10  duration          66 non-null     int64 
 11  publish_time      66 non-null     int64 
 12  song_name_unique  66 non-null     object
 13  publish_date      66 non-null     object
dtypes: int64(4), object(10)
memory usage: 7.7+ KB


In [33]:
clear_and_save_lyric(file_path_prefix, song_data_cleared)

# 采集后，修改模型后，重新处理数据

In [212]:
file_path_prefix = "data/jaychou/"
# file_path_prefix = "data/mayday/"

In [213]:
song_data_cleared_read = pd.read_csv(file_path_prefix +
                                     'cleared_song_data.csv')
clear_and_save_lyric(file_path_prefix, song_data_cleared_read)

In [169]:
song_data_cleared_read

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,tv_name,song_name_unique,publish_date,is_ost
0,629395245,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,风过留痕 影视原声带,81722850,003Dq7Cs1NdHv8,265,1770084000,风过留痕,荣光,2026-02-03,1
1,578702351,001QwWqP38bXCs,烽月,电视剧《折腰》情感主题曲/片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,折腰 影视原声专辑,67523527,001NFTqM3zNUv2,265,1747411200,折腰,烽月,2025-05-17,1
2,370388317,001l38Gx3sl3kP,寻一个你,《苍兰诀》电视剧温情主题曲,刘宇宁,2241311,001Iu4Dv1NzRCD,苍兰诀 东方幻想影视原声带,33455648,001c5tj84NH6dH,267,1660010400,苍兰诀,寻一个你,2022-08-09,1
3,298883373,002kOvBZ3eQChP,天问,《山河令》网剧主题曲,刘宇宁,2241311,001Iu4Dv1NzRCD,山河令 网剧音乐原声大碟,18540389,003HQCiV10f8Bw,221,1614009600,山河令,天问,2021-02-23,1
4,285903302,004a6xSN489klN,当遇见你,《冰糖炖雪梨》电视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,当遇见你,15738588,0027ZeN63fdE93,191,1583769600,冰糖炖雪梨,当遇见你,2020-03-10,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,401100858,0009Ba6M3Rd0kp,望道,《望道》电影同名主题曲,"刘宇宁,8点组乐团","2241311,1162539","001Iu4Dv1NzRCD,004OkF1U0FkWIb",望道,36285879,001SGs7l1BP05L,196,1678982400,望道,望道,2023-03-17,1
138,335548330,0032iLRJ3YH8fN,风起时再见,《迷雾追踪》影视剧主题曲,刘宇宁,2241311,001Iu4Dv1NzRCD,迷雾追踪 影视原声带,33427051,003V3TEN24Hj29,260,1607616000,迷雾追踪,风起时再见,2020-12-11,1
139,264776262,002rs2Tk3fYVGh,要替我幸福,《暖暖，请多指教》电视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,暖暖，请多指教 影视原声带,12373795,003eJOh54YBBK1,242,1589472000,暖暖，请多指教,要替我幸福,2020-05-15,1
140,297559879,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,斗罗大陆 史兰客七怪音乐专辑,17500078,000uo5Ig1624LG,272,1613354400,斗罗大陆,苍穹之下,2021-02-15,1
